In [ ]:
# 1. Secrets 확인
for k in dbutils.secrets.list("dt4_team1_secrets"):
    print(f"  {k.key}")

In [ ]:
# 2. disinfection JSON 읽기
BASE_PATH = "abfss://raw@dt4team1blob.dfs.core.windows.net/raw"
df_disinfection = spark.read.option("multiLine", "true").json(f"{BASE_PATH}/disinfection/")
print(f"rows: {df_disinfection.count()}, columns: {len(df_disinfection.columns)}")
display(df_disinfection.limit(5))

In [ ]:
from pyspark.sql.functions import col, concat, lit, to_date

mapped_df = df_disinfection.select(
    concat(lit("FAC-"), col("dt"), lit("-"), col("ROW_NUM").cast("string")).alias("facility_id"),
    col("ROW_NUM").cast("string").alias("source_seq"),
    col("PSTN_DSNF_PLC_NM").alias("name"),
    col("ADDR").alias("address"),
    col("LAT").cast("double").alias("lat"),
    col("LOT").cast("double").alias("lng"),
    col("OPER_HR").alias("operating_hours"),
    col("PIC_TELNO").alias("phone"),
    lit(None).cast("string").alias("region_code"),
    lit(True).alias("is_active"),
    to_date(col("dt"), "yyyyMM").alias("source_date")
    # created_at, updated_at → DB DEFAULT now()로 자동 처리
)

display(mapped_df.limit(5))
print(f"총 {mapped_df.count()}건")

In [ ]:
pg_host     = dbutils.secrets.get("dt4_team1_secrets", "pg_host")
pg_port     = dbutils.secrets.get("dt4_team1_secrets", "pg_port")
pg_database = dbutils.secrets.get("dt4_team1_secrets", "pg_database")
pg_user     = dbutils.secrets.get("dt4_team1_secrets", "pg_user")
pg_password = dbutils.secrets.get("dt4_team1_secrets", "pg_password")

mapped_df.write.format("postgresql") \
    .option("host", pg_host) \
    .option("port", pg_port) \
    .option("database", pg_database) \
    .option("dbtable", "public.disinfection_facilities") \
    .option("user", pg_user) \
    .option("password", pg_password) \
    .mode("append") \
    .save()

print("✓ disinfection_facilities 적재 완료")